# SAE Scoping Mini-project: End-to-End Experiment (Steps 1–7)

This notebook walks through the full experiment described in the README, using the
`scripts/` CLI tools wherever they already exist and inline Python for the parts
that do not have a standalone script yet.

| Step | Description | How |
|------|-------------|-----|
| 1 | Create a fork | Pre-requisite – do this on GitHub before opening this notebook |
| 2 | Install the package and verify the code works | `pip install -e .` + import check |
| 3 | Pick an in-domain dataset | Config variable + explanation |
| 4 | Calculate & plot SAE firing rates | `!python scripts/find_firing_rates.py` + `!python scripts/plot_firing_rates.py` |
| 5 | Evaluate SAE-hooked model (vary K) | Inline Python using `sae_scoping` |
| 6 | Train the LLM post-SAE-pruning | `!python scripts/train_with_firing_rates.py` |
| 7 | Evaluate finetuned model in-domain & OOD | Inline Python using `sae_scoping` |


---
## Step 1 – Fork the Repository

Pre-requisite – completed before running this notebook.

1. Fork `4gatepylon/SAEScopingMiniproject` on GitHub.
2. Clone **your** fork and open this notebook from inside the repo root:
   ```bash
   git clone <your-fork-url>
   cd SAEScopingMiniproject
   jupyter lab
   ```

---
## Step 2 – Installation & Sanity Check

Install the package and verify that all core imports work.

In [ ]:
# Install the package in editable mode (run once; safe to re-run)
!pip install -e . -q

In [ ]:
# Quick import sanity check
import torch
from transformers import AutoTokenizer, Gemma2ForCausalLM
from sae_lens import SAE
from sae_scoping.trainers.sae_enhanced.rank import rank_neurons
from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.trainers.sae_enhanced.train import train_sae_enhanced_model
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

print(f"torch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
print("All imports OK.")

---
## Step 3 – Pick an In-Domain Dataset

We need **one** in-domain task dataset for scoping.  
Available options from [`4gate/StemQAMixture`](https://huggingface.co/datasets/4gate/StemQAMixture):

| Config | Notes |
|--------|-------|
| `chemistry` | Non-biology STEM – good default |
| `biology` | Adriano's reference run (helpful for comparison but duplicates existing work) |
| `physics` | Another option |

**Choice:** `chemistry` (non-biology STEM, avoids duplicating prior work while staying close enough for easy comparison).

> **Train/test split:** `StemQAMixture` ships with `train` and `test` splits. We use `train` for
> firing-rate estimation **and** SFT training, and hold out `test` for in-domain evaluation.  
> Using the same data for firing-rate estimation and training is acceptable here because
> firing rates only determine *which neurons to keep*, not their weights – the SAE is
> never updated. However, if you want a stricter setup you can hold out a separate
> firing-rate split.

In [ ]:
# ── Configurable variables used by the rest of this notebook ──────────────────
DATASET_NAME        = "chemistry"          # HuggingFace config name for 4gate/StemQAMixture
SCRIPT_DATASET_NAME = "stemqa_chemistry"   # internal key used by find_firing_rates.py
MODEL_NAME          = "google/gemma-2-9b-it"
SAE_LAYER           = 20                   # which Gemma-2 layer to attach the SAE to
SAE_WIDTH           = "16k"
SAE_ID              = f"layer_{SAE_LAYER}/width_{SAE_WIDTH}/canonical"
SAE_RELEASE         = "gemma-scope-9b-pt-res-canonical"
BATCH_SIZE          = 7                    # reduce if OOM
CACHE_DIR           = "scripts/.cache"     # where find_firing_rates.py writes results

print(f"In-domain dataset : {DATASET_NAME} (script key: {SCRIPT_DATASET_NAME})")
print(f"Model             : {MODEL_NAME}")
print(f"SAE               : {SAE_ID} ({SAE_RELEASE})")

---
## Step 4 – Calculate & Plot SAE Firing Rates

### 4a – Calculate firing rates

`scripts/find_firing_rates.py` iterates over every (dataset, ignore-padding, SAE-ID) combination
and writes a `distribution.safetensors` file under `scripts/.cache/`.  
Results are cached, so re-running is cheap.

In [ ]:
# Calculate firing rates for the in-domain dataset (and ultrachat as an OOD reference).
# Flags:
#   -d  comma-separated dataset names (must match internal keys in find_firing_rates.py:
#       stemqa_chemistry | apps | ultrachat)
#   -i  ignore padding tokens (True = cleaner rates; False = include padding)
#   -b  batch size
!python scripts/find_firing_rates.py \
    -d {SCRIPT_DATASET_NAME},ultrachat \
    -i True,False \
    -b {BATCH_SIZE}

### 4b – Plot firing rates

`scripts/plot_firing_rates.py` reads the cached distributions and writes PNG plots to
`scripts/.cache/plots/`.  Generates:
- **Sorted firing rates** – useful for picking a pruning threshold
- **Cumulative firing rates** – shows how many features capture X% of total activity
- **Cross-dataset overlap** – highlights which top-K features are domain-specific

In [ ]:
# Plot the distributions (ignore_padding=True is the cleaner setting)
!python scripts/plot_firing_rates.py \
    --cache-dir scripts/.cache \
    --ignore-padding True \
    --top-k 500

In [ ]:
# Display plots inline
import glob
from IPython.display import Image, display

plot_paths = sorted(glob.glob("scripts/.cache/plots/*.png"))
if not plot_paths:
    print("No plots found – run Step 4a first.")
for p in plot_paths:
    print(p)
    display(Image(filename=p, width=800))

### 4c – Choose a pruning threshold

Look at the **sorted firing-rate** plots above.  Pick a threshold `T` such that the
number of retained neurons is small but the in-domain cumulative firing rate is still
high (≥ 90 %).

A good starting point is `T = 1e-4` (neurons that fire on > 0.01 % of tokens).  
Update `THRESHOLD` below based on your reading of the plots.

In [ ]:
import re
from pathlib import Path
from safetensors.torch import load_file

# ── Set your chosen threshold here ────────────────────────────────────────────
THRESHOLD = 1e-4

# Load the chemistry distribution for the layer we will use
sae_folder = SAE_ID.replace("/", "--")
dist_path = Path(CACHE_DIR) / "ignore_padding_True" / SCRIPT_DATASET_NAME / sae_folder / "distribution.safetensors"
if dist_path.exists():
    dist = load_file(str(dist_path))["distribution"]
    n_kept = (dist >= THRESHOLD).sum().item()
    print(f"Threshold {THRESHOLD:.2e}  →  keep {n_kept} / {len(dist)} neurons "
          f"({100 * n_kept / len(dist):.1f} %)")
else:
    print(f"Distribution file not found at {dist_path}.  Run Step 4a first.")

---
## Step 5 – Evaluate the SAE-Hooked Model (Vary K)

Before training, we check how much performance degrades as we reduce the number of
retained SAE neurons.  This helps us pick the lowest K that still preserves in-domain
capability (the goal: be as sparse as possible while staying useful in-domain).

We measure **perplexity** on a held-out test split as a proxy for performance.  
Lower perplexity = better language-model quality.

> **Note:** This cell loads the full 9 B model and requires a GPU with ≥ 24 GB VRAM.
> On smaller hardware, reduce `N_EVAL_SAMPLES` or use a smaller SAE width.

In [ ]:
from functools import partial
import math
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, Gemma2ForCausalLM
from sae_lens import SAE
from safetensors.torch import load_file
from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_EVAL_SAMPLES = 50   # increase for more reliable estimates
EVAL_MAX_LEN   = 512

# K values to sweep (as fractions of d_sae)
K_FRACTIONS = [1.0, 0.5, 0.25, 0.1, 0.05]


def compute_perplexity(model, tokenizer, texts, max_length=512, device=device):
    """Compute mean per-token perplexity over a list of strings."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
            input_ids = enc["input_ids"].to(device)
            if input_ids.shape[1] < 2:
                continue
            out = model(input_ids, labels=input_ids)
            n_tokens = input_ids.shape[1] - 1
            total_loss   += out.loss.item() * n_tokens
            total_tokens += n_tokens
    return math.exp(total_loss / total_tokens) if total_tokens > 0 else float("inf")


# ── Load model & tokenizer once ───────────────────────────────────────────────
print("Loading tokenizer…")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model (bfloat16, CPU map then to GPU)…")
model = Gemma2ForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cpu", attn_implementation="eager"
)
model = model.to(device)
model.eval()

# ── Load SAE ──────────────────────────────────────────────────────────────────
print(f"Loading SAE {SAE_ID}…")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
sae = sae.to(device)

hookpoint = f"model.layers.{SAE_LAYER}"

# ── Load distribution ─────────────────────────────────────────────────────────
dist = load_file(str(dist_path))["distribution"]   # uses dist_path from Step 4c
neuron_ranking = torch.argsort(dist, descending=True)
d_sae = len(dist)

# ── Evaluation texts ──────────────────────────────────────────────────────────
print("Loading evaluation data…")
in_domain_data  = load_dataset("4gate/StemQAMixture", DATASET_NAME, split="test")
ood_data        = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")

in_domain_texts = in_domain_data["question"][:N_EVAL_SAMPLES]
ood_texts       = [
    next((m["content"] for m in msgs if m["role"] == "user"), "")
    for msgs in ood_data["messages"][:N_EVAL_SAMPLES]
]

print(f"Evaluating with {N_EVAL_SAMPLES} samples per dataset…")

In [ ]:
results = []  # list of dicts for the summary table

# ── Baseline: no SAE hook ─────────────────────────────────────────────────────
print("[baseline] No SAE hook")
ppl_id  = compute_perplexity(model, tokenizer, in_domain_texts, EVAL_MAX_LEN)
ppl_ood = compute_perplexity(model, tokenizer, ood_texts, EVAL_MAX_LEN)
results.append({"K": "baseline (no SAE)", "ppl_in_domain": ppl_id, "ppl_ood": ppl_ood})
print(f"  in-domain PPL={ppl_id:.2f}  OOD PPL={ppl_ood:.2f}")

# ── Sweep K ───────────────────────────────────────────────────────────────────
for k_frac in K_FRACTIONS:
    K = max(1, int(k_frac * d_sae))
    print(f"[K={K} / {d_sae}  ({100*k_frac:.0f}%)]")
    pruned = get_pruned_sae(sae, neuron_ranking, K_or_p=K, T=0.0).to(device)
    sw = SAEWrapper(pruned)
    hook_dict = {hookpoint: partial(filter_hook_fn, sw)}
    with named_forward_hooks(model, hook_dict):
        ppl_id  = compute_perplexity(model, tokenizer, in_domain_texts, EVAL_MAX_LEN)
        ppl_ood = compute_perplexity(model, tokenizer, ood_texts, EVAL_MAX_LEN)
    results.append({"K": K, "k_frac": k_frac, "ppl_in_domain": ppl_id, "ppl_ood": ppl_ood})
    print(f"  in-domain PPL={ppl_id:.2f}  OOD PPL={ppl_ood:.2f}")

# ── Summary table ─────────────────────────────────────────────────────────────
import pandas as pd
df = pd.DataFrame(results)
print("\n=== Step 5 summary ===")
print(df.to_string(index=False))

### 5b – Choose K for training

From the table above, pick the **smallest K** (or threshold) where in-domain PPL
is still acceptably close to the baseline.  Update `THRESHOLD` or `K_TRAIN` below.

In [ ]:
# Update with your choice from the sweep above
K_TRAIN = max(1, int(0.1 * d_sae))  # example: top 10% of neurons
print(f"Training will use K={K_TRAIN} neurons ({100 * K_TRAIN / d_sae:.1f}% of {d_sae}).")

---
## Step 6 – Train the LLM Post-SAE-Pruning

`scripts/train_with_firing_rates.py` loads the pruned SAE (via the distribution file
and a threshold), hooks it into Gemma-2, freezes layers **before** the hookpoint,
and runs SFT on the in-domain dataset.

Key flags:
| Flag | Meaning |
|------|---------|
| `-p` | Path to `distribution.safetensors` from Step 4a |
| `-h` | Firing-rate threshold (keeps neurons with rate ≥ h) |
| `-t` | Dataset name to train on (must match key inside the script) |
| `-b` | Per-device batch size |
| `-a` | Gradient accumulation steps (effective batch = b × a) |
| `-s` | Max training steps |
| `-w` | W&B project name |

> **Note:** Set your W&B API key before running:  
> `import os; os.environ["WANDB_API_KEY"] = "<your-key>"` or `!wandb login`

In [ ]:
import os
# Uncomment and fill in your W&B key if not already logged in
# os.environ["WANDB_API_KEY"] = "<your-wandb-api-key>"

# Derive an effective threshold from K_TRAIN (neurons with firing rate >= threshold)
# Alternatively you can pass the threshold directly with -h
from safetensors.torch import load_file as st_load
from pathlib import Path

_dist = st_load(str(dist_path))["distribution"]
_sorted, _ = _dist.sort(descending=True)
effective_threshold = float(_sorted[K_TRAIN - 1].item())
print(f"Effective threshold for K={K_TRAIN}: {effective_threshold:.2e}")

In [ ]:
# Run SFT training via the CLI script
# Adjust -b/-a/-s to match your GPU memory and desired run length.
!python scripts/train_with_firing_rates.py \
    -p {dist_path} \
    -h {effective_threshold} \
    -t {DATASET_NAME} \
    -b 2 \
    -a 8 \
    -s 1000 \
    -w sae-scoping-experiment \
    --save-every 500 \
    --save-limit 5

After training, note the checkpoint path printed at the end (something like
`outputs_gemma9b/chemistry/layer_20_width_16k_canonical_h<threshold>_<hash>/checkpoint-1000`).
Set `CHECKPOINT_PATH` below for evaluation.

In [ ]:
# ── Set the checkpoint path produced by Step 6 ────────────────────────────────
CHECKPOINT_PATH = None  # e.g. "outputs_gemma9b/chemistry/layer_20_.../checkpoint-1000"

if CHECKPOINT_PATH is None:
    # Auto-detect the latest checkpoint if only one run exists
    import glob as _glob
    candidates = sorted(_glob.glob(f"outputs_gemma9b/{DATASET_NAME}/**/checkpoint-*", recursive=True))
    if candidates:
        CHECKPOINT_PATH = candidates[-1]
        print(f"Auto-detected checkpoint: {CHECKPOINT_PATH}")
    else:
        print("No checkpoint found. Run Step 6 first.")

---
## Step 7 – Evaluate the Finetuned SAE-Hooked Model

We evaluate:
- **In-domain** (`chemistry` test split) – should be good (low PPL, high accuracy)
- **OOD task 1** – `ultrachat` (general conversation) – should be worse than baseline
- **OOD task 2** – `codeparrot/apps` (code generation) – should be worse than baseline

These two OOD tasks were chosen because:
1. *UltraChat* captures general assistant capabilities – the most obvious dual-use surface.
2. *APPS* captures coding – a distinct OOD domain with different vocabulary & reasoning.

By showing degraded performance on *both* we build a stronger argument that scoping is
not task-specific but broadly capability-limiting OOD.

In [ ]:
import math
import torch
from functools import partial
from datasets import load_dataset
from transformers import AutoTokenizer, Gemma2ForCausalLM
from sae_lens import SAE
from safetensors.torch import load_file
from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_EVAL = 100   # increase for more reliable results


def compute_perplexity(model, tokenizer, texts, max_length=512, device=device):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
            input_ids = enc["input_ids"].to(device)
            if input_ids.shape[1] < 2:
                continue
            out = model(input_ids, labels=input_ids)
            n_tokens = input_ids.shape[1] - 1
            total_loss   += out.loss.item() * n_tokens
            total_tokens += n_tokens
    return math.exp(total_loss / total_tokens) if total_tokens > 0 else float("inf")


# ── Load finetuned model from checkpoint ─────────────────────────────────────
if CHECKPOINT_PATH is None:
    raise RuntimeError("Set CHECKPOINT_PATH in Step 6 before running this cell.")

print(f"Loading finetuned model from {CHECKPOINT_PATH}…")
tokenizer_ft = AutoTokenizer.from_pretrained(MODEL_NAME)
model_ft = Gemma2ForCausalLM.from_pretrained(
    CHECKPOINT_PATH, torch_dtype=torch.bfloat16, device_map="cpu", attn_implementation="eager"
)
model_ft = model_ft.to(device)
model_ft.eval()

# ── Load pruned SAE ───────────────────────────────────────────────────────────
print(f"Loading SAE and pruning to K={K_TRAIN}…")
sae_ft = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
sae_ft = sae_ft.to(device)
dist_ft = load_file(str(dist_path))["distribution"]
neuron_ranking_ft = torch.argsort(dist_ft, descending=True)
pruned_ft = get_pruned_sae(sae_ft, neuron_ranking_ft, K_or_p=K_TRAIN, T=0.0).to(device)
hookpoint_ft = f"model.layers.{SAE_LAYER}"
hook_dict_ft = {hookpoint_ft: partial(filter_hook_fn, SAEWrapper(pruned_ft))}

# ── Evaluation datasets ───────────────────────────────────────────────────────
print("Loading evaluation datasets…")
in_domain  = load_dataset("4gate/StemQAMixture", DATASET_NAME, split="test")["question"][:N_EVAL]
ood_chat   = [
    next((m["content"] for m in msgs if m["role"] == "user"), "")
    for msgs in load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")["messages"][:N_EVAL]
]
ood_code   = load_dataset("codeparrot/apps", split="test")["question"][:N_EVAL]

eval_sets = {
    "in-domain (chemistry)": in_domain,
    "OOD – ultrachat"      : ood_chat,
    "OOD – apps (code)"    : ood_code,
}
print(f"Evaluating on {N_EVAL} samples per split…")

In [ ]:
eval_results = []

for split_name, texts in eval_sets.items():
    # Finetuned + pruned SAE (the scoped model)
    with named_forward_hooks(model_ft, hook_dict_ft):
        ppl_scoped = compute_perplexity(model_ft, tokenizer_ft, texts)

    # Baseline: original model without any SAE
    ppl_base = compute_perplexity(model, tokenizer, texts)

    eval_results.append({
        "split"       : split_name,
        "ppl_baseline": ppl_base,
        "ppl_scoped"  : ppl_scoped,
        "delta"       : ppl_scoped - ppl_base,
    })
    print(f"{split_name:35s}  baseline={ppl_base:.2f}  scoped={ppl_scoped:.2f}  Δ={ppl_scoped - ppl_base:+.2f}")

import pandas as pd
df7 = pd.DataFrame(eval_results)
print("\n=== Step 7 summary ===")
print(df7.to_string(index=False))
print("""
Interpretation guide:
  in-domain  Δ ≈ 0   → scoping preserved in-domain performance  ✓
  OOD        Δ > 0   → scoped model is worse OOD (desired)      ✓
  OOD        Δ < 0   → scoped model accidentally improved OOD   ✗
""")

---
## Summary

| Step | Status | Key output |
|------|--------|------------|
| 1 | ✅ pre-req | Fork on GitHub |
| 2 | ✅ | Package installed, imports OK |
| 3 | ✅ | Dataset: `chemistry` (non-biology STEM) |
| 4 | ✅ | Firing rates in `scripts/.cache/`; plots in `scripts/.cache/plots/` |
| 5 | ✅ | K sweep table; chosen `K_TRAIN` |
| 6 | ✅ | Checkpoint in `outputs_gemma9b/` |
| 7 | ✅ | Evaluation table: in-domain PPL preserved, OOD PPL increased |

### Next steps
- **Step 8 (Bonus):** Implement a pruning or unlearning baseline.
- **Step 9 (Bonus):** Argue why your baseline is SoTA.
- **Step 10:** Write up results in Overleaf (≤ 2 pages + appendix).
- **Step 11 (Bonus):** Add Gemma-3 / Gemmascope-2 experiment.